# EDA: подготовка данных перед Feature Engineering

Ноутбук построен так, чтобы результаты EDA можно было напрямую использовать на следующем этапе:
в конце сохраняются очищенные датасеты (`X_clean_v1`, `X_clean_v2`), группы признаков и сводная
таблица `feature_summary` в папку `artifacts/`.

**Структура:**
1. Setup и загрузка
2. Базовая валидация (shapes, ID, target)
3. Типы данных и пропуски
4. Типология признаков (константы, бинарные, low-card, sparse, −1-encoded, skewed)
5. Связь признаков с target (корреляция, биннинг, эффект −1, эффект ненулевого значения)
6. Мультиколлинеарность
7. Стабильность train vs test
8. Итоговая сводка `feature_summary`
9. Cleaning: `X_clean_v1`, `X_clean_v2`
10. Сохранение артефактов для Feature Engineering


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---- Пути ----
DATA_DIR = Path(".")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# ---- Роли колонок ----
TARGET = "target"
ID_COL = "index"

# ---- Пороги EDA (все в одном месте, легко менять) ----
LOW_CARD_MAX_UNIQUE   = 5      # признак считается low-cardinality, если <= 5 уникальных
HIGH_ZERO_MIN_SHARE   = 0.80   # признак считается sparse, если >= 80% нулей
HIGH_SKEW_MIN_ABS     = 5.0    # признак считается skewed, если |skew| >= 5
MINUS_ONE_MIN_PCT     = 1.0    # -1 считаем значимым, если встречается в >= 1% строк
MINUS_ONE_RATE_DIFF   = 0.002  # минимальный сдвиг target rate для "информативного" -1
HIGH_CORR_PAIR_THRESH = 0.90   # порог для поиска скоррелированных пар
TOP_N_FEATURES        = 20     # сколько top-признаков рассматривать детально


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train.shape)
print("Test  shape:", test.shape)
train.head()

## 2. Базовая валидация
Проверяем согласованность train/test, уникальность идентификатора и распределение target.

In [ ]:
only_in_train = [c for c in train.columns if c not in test.columns]
only_in_test  = [c for c in test.columns  if c not in train.columns]

print("Только в train:", only_in_train)
print("Только в test :", only_in_test)
print()
print("index уникален в train:", train[ID_COL].is_unique)
print("index уникален в test :", test[ID_COL].is_unique)
print()
print("Полные дубликаты строк в train:", train.duplicated().sum())
print("Полные дубликаты строк в test :", test.duplicated().sum())
print("Дубликаты без index в train   :", train.drop(columns=[ID_COL]).duplicated().sum())
print("Дубликаты без index в test    :", test.drop(columns=[ID_COL]).duplicated().sum())

In [ ]:
# Распределение target
print("Уникальных значений target:", train[TARGET].nunique())
print()
print(train[TARGET].value_counts(dropna=False))
print()
print(train[TARGET].value_counts(normalize=True, dropna=False).round(4))

plt.figure(figsize=(5, 3))
train[TARGET].value_counts().sort_index().plot(kind="bar")
plt.title("Распределение target")
plt.xlabel("target"); plt.ylabel("count")
plt.tight_layout(); plt.show()

## 3. Типы данных и пропуски
`missing_cnt` и `missing_pct` оставлены на случай появления NaN после будущих преобразований.

In [ ]:
print(train.dtypes.value_counts())

info_df = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "nunique": train.nunique(dropna=False),
    "missing_cnt": train.isna().sum(),
    "missing_pct": train.isna().mean() * 100,
}).sort_values("nunique")

info_df.head(15)

## 4. Типология признаков
Выделяем группы, которые по-разному обрабатываются при FE:
константные, бинарные, low-cardinality, с `-1`, sparse (много нулей), скошенные, регулярные числовые.

In [ ]:
# Базовые признаки (без id и target)
base_feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET]]
X_base      = train[base_feature_cols].copy()
X_test_base = test[base_feature_cols].copy()
y           = train[TARGET].copy()

# Константные признаки
constant_cols = [c for c in base_feature_cols if X_base[c].nunique(dropna=False) == 1]

# Рабочий список признаков (без констант)
feature_cols = [c for c in base_feature_cols if c not in constant_cols]

print("Всего базовых признаков:", len(base_feature_cols))
print("Константных            :", len(constant_cols))
print("После удаления констант:", len(feature_cols))
print()
print("Примеры константных:", constant_cols[:10])

In [ ]:
# Группы признаков
binary_cols      = [c for c in feature_cols if X_base[c].nunique(dropna=False) == 2]
low_card_cols    = [c for c in feature_cols if X_base[c].nunique(dropna=False) <= LOW_CARD_MAX_UNIQUE]
minus_one_cols   = [c for c in feature_cols if (X_base[c] == -1).any()]
high_zero_cols   = [c for c in feature_cols if (X_base[c] == 0).mean() >= HIGH_ZERO_MIN_SHARE]

skew_all = X_base[feature_cols].skew()
high_skew_cols = skew_all[skew_all.abs() >= HIGH_SKEW_MIN_ABS].index.tolist()

# Регулярные числовые — то, что не попало ни в одну из "особых" групп
special = set(low_card_cols + minus_one_cols + high_zero_cols + high_skew_cols)
regular_numeric_cols = [c for c in feature_cols if c not in special]

print(f"binary_cols          : {len(binary_cols)}")
print(f"low_card_cols (<= {LOW_CARD_MAX_UNIQUE}): {len(low_card_cols)}")
print(f"minus_one_cols       : {len(minus_one_cols)}")
print(f"high_zero_cols (>= {int(HIGH_ZERO_MIN_SHARE*100)}%): {len(high_zero_cols)}")
print(f"high_skew_cols (|skew|>={HIGH_SKEW_MIN_ABS}): {len(high_skew_cols)}")
print(f"regular_numeric_cols : {len(regular_numeric_cols)}")

In [ ]:
# Описательная статистика и признаки с подозрительным масштабом
desc = X_base[feature_cols].describe().T
desc["range"] = desc["max"] - desc["min"]

# Сортируем по диапазону — на верх всплывают признаки с большими значениями / выбросами
desc.sort_values("range", ascending=False).head(15)

In [ ]:
# Доминирование одного значения
dominance_df = pd.DataFrame({
    "top_value_share": X_base[feature_cols].apply(
        lambda s: s.value_counts(normalize=True, dropna=False).iloc[0]
    ),
    "nunique":        X_base[feature_cols].nunique(dropna=False),
    "zero_pct":       (X_base[feature_cols] == 0).mean() * 100,
    "minus_one_pct":  (X_base[feature_cols] == -1).mean() * 100,
}).sort_values("top_value_share", ascending=False)

dominance_df.head(20)

## 5. Связь признаков с target
Считаем линейную корреляцию, эффект `-1`, эффект ненулевого значения и бининг для top-признаков.

In [ ]:
# corrwith работает столбец-к-столбцу, без построения полной матрицы 1367x1367
corr_with_target = X_base[feature_cols].corrwith(y)
corr_with_target_abs = corr_with_target.abs().sort_values(ascending=False)

top_corr_features = corr_with_target_abs.head(TOP_N_FEATURES).index.tolist()

corr_with_target_abs.head(TOP_N_FEATURES).to_frame("abs_corr")

In [ ]:
# Эффект -1 на target rate
rows = []
for col in minus_one_cols:
    is_minus_one = (X_base[col] == -1)
    n = is_minus_one.sum()
    if n == 0:
        continue
    rate_m1   = y[is_minus_one].mean()
    rate_rest = y[~is_minus_one].mean()
    rows.append({
        "feature": col,
        "minus_one_cnt": int(n),
        "minus_one_pct": is_minus_one.mean() * 100,
        "rate_when_m1":  rate_m1,
        "rate_when_not": rate_rest,
        "rate_diff":     rate_m1 - rate_rest,
    })

minus_one_effect_df = pd.DataFrame(rows).sort_values("rate_diff", ascending=False)

# Отмечаем признаки, где -1 информативен
minus_one_effect_df["informative_m1"] = (
    (minus_one_effect_df["minus_one_pct"] >= MINUS_ONE_MIN_PCT) &
    (minus_one_effect_df["rate_diff"].abs() > MINUS_ONE_RATE_DIFF)
)

informative_minus_one_cols = minus_one_effect_df.loc[
    minus_one_effect_df["informative_m1"], "feature"
].tolist()

print("Признаков, где -1 информативен:", len(informative_minus_one_cols))
minus_one_effect_df.head(15)

In [ ]:
# Эффект ненулевого значения (для sparse-признаков)
rows = []
for col in high_zero_cols:
    is_nonzero = (X_base[col] != 0)
    n = is_nonzero.sum()
    if n == 0:
        continue
    rows.append({
        "feature": col,
        "nonzero_pct":   is_nonzero.mean() * 100,
        "rate_nonzero":  y[is_nonzero].mean(),
        "rate_zero":     y[~is_nonzero].mean(),
    })

nonzero_effect_df = pd.DataFrame(rows)
nonzero_effect_df["rate_diff"] = (
    nonzero_effect_df["rate_nonzero"] - nonzero_effect_df["rate_zero"]
)
nonzero_effect_df = nonzero_effect_df.sort_values(
    "rate_diff", key=lambda s: s.abs(), ascending=False
)

nonzero_effect_df.head(15)

In [ ]:
# Event rate по квантильным бинам для top-признаков (сетка вместо N отдельных фигур)
n_plot = min(6, len(top_corr_features))
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, col in enumerate(top_corr_features[:n_plot]):
    tmp = pd.DataFrame({"x": X_base[col], "y": y})
    try:
        tmp["bin"] = pd.qcut(tmp["x"], q=10, duplicates="drop")
        stat = tmp.groupby("bin", observed=True)["y"].mean()
        axes[i].plot(range(len(stat)), stat.values, marker="o")
        axes[i].set_title(col)
        axes[i].set_xlabel("bin"); axes[i].set_ylabel("target rate")
    except Exception as e:
        axes[i].set_title(f"{col} (err)")

for j in range(n_plot, len(axes)):
    axes[j].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
# Boxplot top-признаков в разрезе target (одной сеткой)
n_plot = min(6, len(top_corr_features))
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, col in enumerate(top_corr_features[:n_plot]):
    data = [X_base.loc[y == v, col].values for v in sorted(y.unique())]
    axes[i].boxplot(data, labels=sorted(y.unique()), showfliers=False)
    axes[i].set_title(col)
    axes[i].set_xlabel("target")

for j in range(n_plot, len(axes)):
    axes[j].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
# Event rate по значениям low-cardinality признаков (первые 10)
for col in low_card_cols[:10]:
    stat = (
        pd.DataFrame({"v": X_base[col], "y": y})
        .groupby("v")["y"]
        .agg(n_obs="count", target_rate="mean")
    )
    print(f"--- {col} ---")
    print(stat)
    print()

## 6. Мультиколлинеарность
Ищем пары признаков с `|corr| >= HIGH_CORR_PAIR_THRESH`. Считаем корреляцию только среди top-признаков,
чтобы не строить матрицу на все ~1350 колонок.

In [ ]:
# Расширяем top до 100 наиболее коррелирующих с target — этого хватит для поиска групп
top_for_corr = corr_with_target_abs.head(100).index.tolist()

corr_matrix = X_base[top_for_corr].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .loc[lambda s: s >= HIGH_CORR_PAIR_THRESH]
    .sort_values(ascending=False)
    .to_frame("abs_corr")
)

print("Пар с |corr| >=", HIGH_CORR_PAIR_THRESH, ":", len(high_corr_pairs))
high_corr_pairs.head(20)

## 7. Стабильность train vs test
Сравниваем основные статистики между train и test — ищем признаки с распределительным сдвигом.

In [ ]:
compare_df = pd.DataFrame({
    "train_mean":       X_base.mean(),
    "test_mean":        X_test_base.mean(),
    "train_std":        X_base.std(),
    "test_std":         X_test_base.std(),
    "train_zero_pct":   (X_base == 0).mean() * 100,
    "test_zero_pct":    (X_test_base == 0).mean() * 100,
    "train_minus1_pct": (X_base == -1).mean() * 100,
    "test_minus1_pct":  (X_test_base == -1).mean() * 100,
})
compare_df["mean_diff_abs"]      = (compare_df["train_mean"]       - compare_df["test_mean"]).abs()
compare_df["std_diff_abs"]       = (compare_df["train_std"]        - compare_df["test_std"]).abs()
compare_df["zero_pct_diff_abs"]  = (compare_df["train_zero_pct"]   - compare_df["test_zero_pct"]).abs()
compare_df["minus1_pct_diff_abs"]= (compare_df["train_minus1_pct"] - compare_df["test_minus1_pct"]).abs()

print("Top по сдвигу среднего:")
compare_df.sort_values("mean_diff_abs", ascending=False).head(10)

In [ ]:
print("Top по сдвигу доли -1:")
compare_df.sort_values("minus1_pct_diff_abs", ascending=False).head(10)

## 8. Сводная таблица `feature_summary`
Один датафрейм, в котором для каждого признака собрано всё нужное для решений на этапе FE.

In [ ]:
feature_summary = pd.DataFrame({
    "nunique":         X_base[feature_cols].nunique(),
    "top_value_share": X_base[feature_cols].apply(
        lambda s: s.value_counts(normalize=True, dropna=False).iloc[0]
    ),
    "zero_pct":        (X_base[feature_cols] == 0).mean() * 100,
    "minus_one_pct":   (X_base[feature_cols] == -1).mean() * 100,
    "skew":            X_base[feature_cols].skew(),
    "abs_corr_target": corr_with_target_abs,
    "mean_diff_tt":    compare_df["mean_diff_abs"],
    "minus1_diff_tt":  compare_df["minus1_pct_diff_abs"],
})

# Флаги групп
feature_summary["is_binary"]      = feature_summary.index.isin(binary_cols)
feature_summary["is_low_card"]    = feature_summary.index.isin(low_card_cols)
feature_summary["has_minus_one"]  = feature_summary.index.isin(minus_one_cols)
feature_summary["is_high_zero"]   = feature_summary.index.isin(high_zero_cols)
feature_summary["is_high_skew"]   = feature_summary.index.isin(high_skew_cols)
feature_summary["is_regular"]     = feature_summary.index.isin(regular_numeric_cols)
feature_summary["informative_m1"] = feature_summary.index.isin(informative_minus_one_cols)

feature_summary = feature_summary.sort_values("abs_corr_target", ascending=False)
feature_summary.head(20)

## 9. Cleaning
Готовим две версии данных:
- **v1**: `-1` сохраняется как есть (базовая версия для baseline).
- **v2**: `-1` заменяется на `NaN` (альтернатива, проверить на baseline позже).

In [ ]:
# v1 — без изменений значений, только без констант
X_clean_v1      = X_base[feature_cols].copy()
X_test_clean_v1 = X_test_base[feature_cols].copy()

# v2 — с -1 -> NaN
X_clean_v2      = X_clean_v1.copy()
X_test_clean_v2 = X_test_clean_v1.copy()
X_clean_v2[minus_one_cols]      = X_clean_v2[minus_one_cols].replace(-1, np.nan)
X_test_clean_v2[minus_one_cols] = X_test_clean_v2[minus_one_cols].replace(-1, np.nan)

print("v1:", X_clean_v1.shape, "| test:", X_test_clean_v1.shape)
print("v2:", X_clean_v2.shape, "| test:", X_test_clean_v2.shape)
print("NaN в v2 (train):", int(X_clean_v2.isna().sum().sum()))
print("NaN в v2 (test) :", int(X_test_clean_v2.isna().sum().sum()))

## 10. Сохранение артефактов
Эти файлы читаются в следующем ноутбуке с Feature Engineering — никаких повторных расчётов `constant_cols`, `minus_one_cols` и т.п. не потребуется.

In [ ]:
# Датасеты
X_clean_v1.to_parquet(ARTIFACTS_DIR / "X_train_v1.parquet")
X_test_clean_v1.to_parquet(ARTIFACTS_DIR / "X_test_v1.parquet")
X_clean_v2.to_parquet(ARTIFACTS_DIR / "X_train_v2.parquet")
X_test_clean_v2.to_parquet(ARTIFACTS_DIR / "X_test_v2.parquet")
y.to_frame().to_parquet(ARTIFACTS_DIR / "y_train.parquet")

# ID'ы (пригодятся для submission)
train[[ID_COL]].to_parquet(ARTIFACTS_DIR / "train_ids.parquet")
test[[ID_COL]].to_parquet(ARTIFACTS_DIR / "test_ids.parquet")

# Группы признаков
feature_groups = {
    "constant":         constant_cols,
    "binary":           binary_cols,
    "low_cardinality":  low_card_cols,
    "minus_one":        minus_one_cols,
    "informative_m1":   informative_minus_one_cols,
    "high_zero":        high_zero_cols,
    "high_skew":        high_skew_cols,
    "regular_numeric":  regular_numeric_cols,
    "top_corr":         top_corr_features,
    "high_corr_pairs":  high_corr_pairs.reset_index().values.tolist(),
}
with open(ARTIFACTS_DIR / "feature_groups.pkl", "wb") as f:
    pickle.dump(feature_groups, f)

# Сводные таблицы
feature_summary.to_parquet(ARTIFACTS_DIR / "feature_summary.parquet")
minus_one_effect_df.to_parquet(ARTIFACTS_DIR / "minus_one_effect.parquet")
nonzero_effect_df.to_parquet(ARTIFACTS_DIR / "nonzero_effect.parquet")
compare_df.to_parquet(ARTIFACTS_DIR / "train_test_compare.parquet")

print("Сохранено в:", ARTIFACTS_DIR.resolve())
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(" -", p.name)

## Финальные правила очистки данных перед baseline

1. `index` исключён из признаков, сохранён отдельно как идентификатор.
2. `target` отделён от матрицы признаков.
3. Константные признаки удалены из train и test.
4. Sparse и low-cardinality признаки **не удаляются автоматически**: часть из них показывает значимый эффект на target (см. `nonzero_effect_df` и таблицу по low-card в разделе 5).
5. Признаки с `-1` сохранены как есть в **v1**; распределение `-1` стабильно между train и test (см. `compare_df`).
6. Версия **v2** с `-1 → NaN` сохранена как альтернатива для сравнения на baseline.
7. Агрессивное удаление скоррелированных признаков на этом этапе **не применяется**: список пар `high_corr_pairs` сохранён, решение принимается после baseline по feature importance.

## Что делать в Feature Engineering

Всё уже подготовлено, достаточно прочитать артефакты:

```python
X = pd.read_parquet("artifacts/X_train_v1.parquet")
y = pd.read_parquet("artifacts/y_train.parquet")["target"]
with open("artifacts/feature_groups.pkl", "rb") as f:
    groups = pickle.load(f)
summary = pd.read_parquet("artifacts/feature_summary.parquet")
```

Направления для FE:
- бинаризация `is_minus_one` для признаков из `informative_m1`;
- бинаризация `is_nonzero` для признаков из `high_zero` с сильным `rate_diff` в `nonzero_effect_df`;
- логарифмирование / winsorization для `high_skew`;
- агрегации / PCA по группам скоррелированных признаков из `high_corr_pairs`;
- target encoding для `low_cardinality`.
